In [1]:
"""Notebook Lens sample cell: stream a large Hugging Face text dataset."""

from __future__ import annotations

import html
import os
import re
import warnings
from collections import Counter
from itertools import islice
from statistics import mean


warnings.filterwarnings("ignore", message="IProgress not found.*")

DATASET_NAME = os.environ.get("HF_TEXT_DATASET", "HuggingFaceFW/fineweb-edu")
DATASET_CONFIG = os.environ.get("HF_TEXT_CONFIG", "sample-10BT") or None
DATASET_SPLIT = os.environ.get("HF_TEXT_SPLIT", "train")
TEXT_FIELD = os.environ.get("HF_TEXT_FIELD", "text")
ROW_LIMIT = int(os.environ.get("HF_TEXT_ROWS", "5000"))


def display_html(markup: str) -> None:
    try:
        from IPython.display import HTML, display
    except ModuleNotFoundError:
        print(re.sub(r"<[^>]+>", " ", markup))
    else:
        display(HTML(markup))


def load_stream():
    try:
        from datasets import load_dataset
    except ModuleNotFoundError:
        print("Missing optional dependency: datasets")
        print("Install with: python -m pip install datasets huggingface_hub")
        return None

    kwargs = {"split": DATASET_SPLIT, "streaming": True}
    if DATASET_CONFIG:
        return load_dataset(DATASET_NAME, DATASET_CONFIG, **kwargs)
    return load_dataset(DATASET_NAME, **kwargs)


stream = load_stream()
if stream is None:
    HF_SAMPLE_STATUS = "missing optional dependency"
else:
    lengths: list[int] = []
    source_counts: Counter[str] = Counter()
    token_counts: Counter[str] = Counter()
    examples: list[tuple[int, str, str]] = []

    for row_number, row in enumerate(islice(stream, ROW_LIMIT), start=1):
        text = str(row.get(TEXT_FIELD) or "")
        lengths.append(len(text))

        source = row.get("source") or row.get("url") or row.get("dump") or "unknown"
        source_counts[str(source)[:80]] += 1

        tokens = re.findall(r"[A-Za-z][A-Za-z0-9_'-]{2,}", text.lower())
        token_counts.update(tokens[:200])

        if len(examples) < 5:
            preview = re.sub(r"\s+", " ", text).strip()[:240]
            examples.append((row_number, str(source)[:80], preview))

    if lengths:
        print(f"dataset: {DATASET_NAME}")
        print(f"config: {DATASET_CONFIG or '(default)'}")
        print(f"split: {DATASET_SPLIT}")
        print(f"rows_streamed: {len(lengths)}")
        print(f"mean_chars: {mean(lengths):.1f}")
        print(f"max_chars: {max(lengths)}")
        print("top_sources:")
        for source, count in source_counts.most_common(8):
            print(f"- {source}: {count}")
        print("top_tokens:")
        for token, count in token_counts.most_common(12):
            print(f"- {token}: {count}")

        rows = "\n".join(
            "<tr>"
            f"<td>{row_number}</td>"
            f"<td>{html.escape(source)}</td>"
            f"<td>{html.escape(preview)}</td>"
            "</tr>"
            for row_number, source, preview in examples
        )
        display_html(
            "<h3>Streaming Text Sample</h3>"
            "<table>"
            "<tr><th>row</th><th>source</th><th>preview</th></tr>"
            f"{rows}"
            "</table>"
        )
        HF_SAMPLE_STATUS = "ok"
    else:
        print("No rows streamed.")
        HF_SAMPLE_STATUS = "empty"



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.11/site-p

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.11/site-p

AttributeError: _ARRAY_API not found

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

dataset: HuggingFaceFW/fineweb-edu
config: sample-10BT
split: train
rows_streamed: 200
mean_chars: 3877.3
max_chars: 65092
top_sources:
- http://austenauthors.net/the-independent-jane: 1
- http://query.nytimes.com/gst/fullpage.html?res=9404E7DA1339F934A25751C0A96E9C8B6: 1
- http://www.childline.org.uk/Explore/SexRelationships/Pages/HIVAIDS.aspx: 1
- http://www.ctt.org/resource_centre/getting_started/learning/understanding_licens: 1
- http://www.environment.ucla.edu/water/news/article.asp?parentid=6178: 1
- http://bayweekly.com/old-site/year99/issue7_40/kids7_40.html: 1
- http://phys.org/news4969.html: 1
- http://www.bbc.co.uk/news/world-latin-america-12292661: 1
top_tokens:
- the: 2639
- and: 1327
- that: 430
- for: 393
- are: 323
- with: 285
- this: 260
- from: 201
- was: 181
- you: 178
- have: 176
- not: 145


row,source,preview
1,http://austenauthors.net/the-independent-jane,"The Independent Jane For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose. Elizabeth’s refusal of Mr. Collins offer of marria"
2,http://query.nytimes.com/gst/fullpage.html?res=9404E7DA1339F934A25751C0A96E9C8B6,"Taking Play Seriously By ROBIN MARANTZ HENIG Published: February 17, 2008 On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children,"
3,http://www.childline.org.uk/Explore/SexRelationships/Pages/HIVAIDS.aspx,"How do you get HIV? HIV can be passed on when infected bodily fluid, such as blood or semen, is passed into an uninfected person. Semen is the liquid which is released from a man's penis during sex which carries sperm. It can be infected wi"
4,http://www.ctt.org/resource_centre/getting_started/learning/understanding_licens,CTComms sends on average 2 million emails monthly on behalf of over 125 different charities and not for profits. Take the complexity of technology and stir in the complexity of the legal system and what do you get? Software licenses! If you
5,http://www.environment.ucla.edu/water/news/article.asp?parentid=6178,Hold the salt: UCLA engineers develop revolutionary new desalination membrane Process uses atmospheric pressure plasma to create filtering 'brush layer' Desalination can become more economical and used as a viable alternate water resource.


# Finding

The bounded FineWeb-Edu stream completed for 200 rows. The sample averaged 3,877.3 characters per row with one very long row at 65,092 characters; the first 200 rows came from distinct source URLs, so the quick source summary was mostly a diversity check rather than a frequency signal.
